In [62]:
import numpy as np
import pandas as pd
from pathlib import Path

In [63]:
# Create CSV mapping ISO 3-letter codes to country names
#  (converts copy-pasted Wikipedia table into useful CSV)

# iso_df = pd.read_csv('iso_name_to_iso3c.csv', header=None)
# for index, row in iso_df.iterrows():
#     split = row[0].split('\xa0\xa0')
#     iso3c = split[0]
#     iso_name = split[1]
#     iso_df.at[index, 'iso3c'] = iso3c
#     iso_df.at[index, 'iso_name'] = iso_name
# iso_df = iso_df.drop(columns=[0])
# iso_df.to_csv('iso3c_to_iso_name.csv', index=False)
# iso_df

In [64]:
##############
### SCHEMA ###
##############

# - output/*/allocations/allocations_wide.csv has pathways of sorts
#   - has pathway for various emissions categories per ISO region
#   - want to map to IMAGE regions
#   - approach: ISO3C --> country name --> IMAGE region
# (idk what each emissions category is)
### 

In [65]:
image_mapping_table = pd.read_csv('IMAGE_region_country_mapping.csv')

IMAGE_COUNTRY_MAP = {}
for _, row in image_mapping_table.iterrows():
    region = row['Region']
    countries = row['Countries'].split(', ')
    countries = [c.split('(')[0].strip() for c in countries]

    for c in countries:
        IMAGE_COUNTRY_MAP[c] = region

In [66]:
iso3c_mapping_table = pd.read_csv('iso3c_to_iso_name.csv')
ISO3C_TO_ISO_NAME = {
    row['iso3c']: row['iso_name'] for _, row in iso3c_mapping_table.iterrows()
}

In [67]:
OUT_PATH = Path('../output/primap-202503_wdi-2025_un-owid-2025_unu-wider-2025_melo-2026_rcb-pathways-exponential-decay_all-ghg/')

In [68]:
results = pd.read_csv(OUT_PATH / 'allocations' / 'reference_pathway_allocations_rcb_pathways' /  'allocations_wide.csv')

C:\Users\sherg\AppData\Local\Temp\2\ipykernel_12944\2449240877.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  results = pd.read_csv(OUT_PATH / 'allocations' / 'reference_pathway_allocations_rcb_pathways' /  'allocations_wide.csv')


In [69]:
# How many countries are unmapped or incorrectly mapped?
failed_countries = results[results['iso3c'].map(ISO3C_TO_ISO_NAME).map(IMAGE_COUNTRY_MAP).isna()]['iso3c'].unique()

for c in failed_countries:
    print(f"Failed to map {c} --> {ISO3C_TO_ISO_NAME.get(c, 'Unknown Country Name')}")

# NOTE for future: I fixed most of these by adjusting the `iso3c_to_iso_name` mapping to use IMAGE 
#  names instead of ISO names (e.g. 'United Kingdom of Great Britain and Northern Ireland' --> 'United Kingdom')

# Left with just 'ROW', which I imagine is "Rest of World" and thus unmappable to an IMAGE region.
# NOTE: Decided to drop ROW from the dataset.
results = results[results['iso3c'] != 'ROW']

Failed to map ROW --> Unknown Country Name


In [ ]:
if 'region' in results.columns:
    # surely mr. gpt can't be right that this is the cleanest way to add a column and change the order this way at the same time
    new_columns = results.columns.tolist()
    new_columns.insert(new_columns.index('iso3c') + 1, 'region')
    results['region'] = results['iso3c'].map(ISO3C_TO_ISO_NAME).map(IMAGE_COUNTRY_MAP)
    results = results[new_columns]

results.head(3)

,source-id,allocation-folder,emissions-source,gdp-source,population-source,gini-source,emission-category,target-source,data-type,approach-short,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
0,primap-202503_wdi-2025_un-owid-2025_unu-wider-...,reference_pathway_allocations_rcb_pathways,primap-202503,wdi-2025,un-owid-2025,unu-wider-2025,co2,rcb-pathways,absolute,CPCC-y2015-rw0.0-cw0.0-cy2050,...,0.493493,0.419400,0.351048,0.287993,0.229824,0.176163,0.126660,0.080992,0.038864,0.0
1,primap-202503_wdi-2025_un-owid-2025_unu-wider-...,reference_pathway_allocations_rcb_pathways,primap-202503,wdi-2025,un-owid-2025,unu-wider-2025,co2,rcb-pathways,absolute,CPCC-y2015-rw0.0-cw0.0-cy2050,...,0.003211,0.002594,0.002067,0.001617,0.001232,0.000903,0.000622,0.000381,0.000176,0.0
2,primap-202503_wdi-2025_un-owid-2025_unu-wider-...,reference_pathway_allocations_rcb_pathways,primap-202503,wdi-2025,un-owid-2025,unu-wider-2025,co2,rcb-pathways,absolute,CPCC-y2015-rw0.0-cw0.0-cy2050,...,0.008018,0.006541,0.005260,0.004151,0.003190,0.002357,0.001636,0.001011,0.000469,0.0


In [71]:
results.to_csv(OUT_PATH / 'allocations' / 'reference_pathway_allocations_rcb_pathways' /  'allocations_wide_mapped.csv', index=False)